# 17 — dim_fornecedor (SCD Type 2)

## Purpose
Build the `dim_fornecedor` dimension table using SCD Type 2 (selective tracking).
Each row represents one version of a supplier. On first load, every CNPJ gets a
single current version. Future loads will detect attribute changes via `_attr_hash`
and insert new versions while closing the previous ones.

## Input
- `local/data/silver/silver_identidade/` — partitioned Parquet by UF
- `local/data/silver/silver_ceis/data.parquet`
- `local/data/silver/silver_cnep/data.parquet`

## Output
- `local/data/gold/dim_fornecedor.parquet`

## Steps
1. Imports and configuration
2. Load silver sources as lazy views
3. Materialise t_dim_base (identity + sanction + hash — single statement)
4. Loop por UF — COPY from t_dim_base to partitioned Parquet
5. Consolidate parts into final Parquet file
6. Validation

## Architecture Notes
- **OOM root cause (documented in DuckDB official docs):** multiple blocking operators
  in the same query (window function + hash join) cause out-of-memory even with disk
  spilling enabled. Solution: separate blocking operators into independent statements.
- **Step 3** materialises the full join + deduplication + hash as a native DuckDB table.
  QUALIFY is used instead of a subquery ROW_NUMBER to keep the plan simpler.
- **Step 4** reads from the native table (already on disk) — no join, no window function.
  Each UF COPY is a simple filter + projection, O(n/28) memory per iteration.
- **supplier_sk** uses MD5(cnpj_normalized || valid_from) — deterministic, idempotent,
  no global ORDER BY required, compatible with distributed execution in Databricks.
- **_attr_hash** uses SHA-256 over pipe-separated tracked fields (ADR-004).
  Field order is an immutable contract — reordering generates false-positive SCD2 versions.
- **Join rule (ADR-003):** never join facts to dim_fornecedor using is_current = True alone.
  Always use temporal join: valid_from <= fact_date AND valid_to > fact_date.
- Idempotent — safe to re-run (overwrites output file).


## Step 1 — Imports and configuration

In [ ]:
import sys
import duckdb
import shutil
from pathlib import Path
from datetime import datetime, timezone, date

# --- Resolve project root ---
def _find_root(start: Path) -> Path:
    current = start if start.is_dir() else start.parent
    for candidate in [current, *current.parents]:
        if (candidate / "utils" / "paths.py").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return start

try:
    _seed = Path(__vsc_ipynb_file__).resolve()
except NameError:
    _seed = Path.cwd().resolve()

PROJECT_ROOT = _find_root(_seed)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import silver_path, gold_path, ensure_dir

# --- Paths ---
SILVER_IDENTIDADE = silver_path(PROJECT_ROOT, "silver_identidade", glob=True)
SILVER_CEIS       = silver_path(PROJECT_ROOT, "silver_ceis")
SILVER_CNEP       = silver_path(PROJECT_ROOT, "silver_cnep")

OUTPUT_PATH     = Path(gold_path(PROJECT_ROOT, "dim_fornecedor"))
OUTPUT_PATH_STR = gold_path(PROJECT_ROOT, "dim_fornecedor")
ensure_dir(OUTPUT_PATH.parent)

# --- SCD2 constants ---
VALID_FROM = date.today().isoformat()
VALID_TO   = "9999-12-31"
LOADED_AT  = datetime.now(timezone.utc).isoformat()

print(f"Silver identidade : {SILVER_IDENTIDADE}")
print(f"Silver ceis       : {SILVER_CEIS}")
print(f"Silver cnep       : {SILVER_CNEP}")
print(f"Output            : {OUTPUT_PATH}")
print(f"valid_from        : {VALID_FROM}")
print(f"loaded_at         : {LOADED_AT}")

## Step 2 — Load silver sources as lazy views

In [ ]:
con = duckdb.connect()

con.execute("""
    CREATE OR REPLACE VIEW v_identidade AS
    SELECT * FROM read_parquet('{silver_id}', hive_partitioning=true)
""".format(silver_id=SILVER_IDENTIDADE))

con.execute("""
    CREATE OR REPLACE VIEW v_ceis AS
    SELECT cnpj_normalized FROM read_parquet('{ceis}')
""".format(ceis=SILVER_CEIS))

con.execute("""
    CREATE OR REPLACE VIEW v_cnep AS
    SELECT cnpj_normalized FROM read_parquet('{cnep}')
""".format(cnep=SILVER_CNEP))

# Sanity check — lazy count, no data loaded yet
counts = con.execute("""
    SELECT
        (SELECT COUNT(*)                        FROM v_identidade) AS identidade,
        (SELECT COUNT(DISTINCT cnpj_normalized) FROM v_ceis)       AS ceis,
        (SELECT COUNT(DISTINCT cnpj_normalized) FROM v_cnep)       AS cnep
""").df()

print(counts.to_string(index=False))

## Step 3 — Materialise t_dim_base

### Why this step exists
DuckDB's official documentation states that **multiple blocking operators in the same
query may cause OOM** even when disk spilling is enabled. In our case:
- `ROW_NUMBER() OVER (PARTITION BY cnpj_normalized)` — blocking (window function)
- Hash join against 70M-row rf_empresas — blocking (hash table build)

Running both in one query caused the OutOfMemoryException observed earlier.

**Solution:** materialise the entire transformation in a single `CREATE TABLE` statement.
DuckDB handles each blocking operator in isolation and spills to disk between them.
The result is a native DuckDB table — subsequent steps read from disk, not from Parquet.

### QUALIFY vs subquery ROW_NUMBER
`QUALIFY` is syntactic sugar that applies a window function filter in the same query
without a subquery wrapper. Functionally equivalent to:
```sql
WITH ranked AS (SELECT *, ROW_NUMBER() OVER (...) AS rn FROM ...)
SELECT ... FROM ranked WHERE rn = 1
```
but with a simpler query plan that gives the optimiser more room to push filters down.


In [ ]:
con.execute("SET threads TO 2")
con.execute("SET memory_limit = '8GB'")
con.execute("SET preserve_insertion_order = false")

print("Materialising t_dim_base — this may take 10-30 minutes...")
print("(window function + join separated from COPY — no combined blocking operators)")
t0 = datetime.now()

con.execute("""
    CREATE OR REPLACE TABLE t_dim_base AS
    WITH identity_base AS (
        -- Deduplication: one row per cnpj_normalized, most recent _silver_loaded_at
        -- QUALIFY applies the window filter without a wrapping subquery
        SELECT
            cnpj_normalized,
            razao_social,
            porte_empresa           AS porte,
            situacao_cadastral_desc AS situacao_cadastral,
            natureza_juridica_desc,
            uf,
            municipio_desc
        FROM v_identidade
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY cnpj_normalized
            ORDER BY _silver_loaded_at DESC
        ) = 1
    ),
    sancoes AS (
        -- UNION (not UNION ALL) deduplicates CNPJs present in both tables
        SELECT cnpj_normalized FROM v_ceis
        UNION
        SELECT cnpj_normalized FROM v_cnep
    )
    SELECT
        b.cnpj_normalized,
        b.razao_social,
        b.porte,
        b.situacao_cadastral,
        b.natureza_juridica_desc,
        b.uf,
        b.municipio_desc,
        CASE WHEN s.cnpj_normalized IS NOT NULL
             THEN TRUE ELSE FALSE END                                AS tem_sancao,
        -- SHA-256 over tracked fields (ADR-004)
        -- Field order is immutable — reordering causes false-positive SCD2 versions
        sha256(
            COALESCE(b.razao_social,           '') || '|' ||
            COALESCE(b.porte,                  '') || '|' ||
            COALESCE(b.situacao_cadastral,     '') || '|' ||
            COALESCE(b.natureza_juridica_desc, '') || '|' ||
            COALESCE(b.uf,                     '') || '|' ||
            COALESCE(CAST(
                CASE WHEN s.cnpj_normalized IS NOT NULL THEN TRUE ELSE FALSE END
            AS VARCHAR), '')
        )::VARCHAR                                                   AS _attr_hash
    FROM identity_base b
    LEFT JOIN sancoes s USING (cnpj_normalized)
""")

n       = con.execute("SELECT COUNT(*) FROM t_dim_base").fetchone()[0]
elapsed = (datetime.now() - t0).total_seconds()
print(f"t_dim_base ready: {n:,} rows in {elapsed:.0f}s ({elapsed/60:.1f} min)")

# Sanity check: sanction distribution
dist = con.execute(
    "SELECT tem_sancao, COUNT(*) AS total FROM t_dim_base GROUP BY 1 ORDER BY 1"
).df()
print()
print(dist.to_string(index=False))

## Step 4 — Loop por UF: COPY from t_dim_base to partitioned Parquet

Each iteration:
- reads only the rows for one UF from the native table (already on disk)
- adds SCD2 metadata columns (supplier_sk, valid_from, valid_to, is_current, _loaded_at)
- writes one Parquet file per UF

No joins. No window functions. Memory footprint per iteration ≈ rows(UF) × row_size.
Worst case (SP ~20M rows × ~200 bytes) ≈ 4 GB — well within the 8 GB limit.

**supplier_sk** — MD5(cnpj_normalized || valid_from):
- deterministic and idempotent: same CNPJ + same date always produces the same key
- no global ORDER BY required: avoids the sort blocking operator entirely
- compatible with distributed execution: Spark can compute MD5 per partition
  without coordinating a global sequence counter across nodes


In [ ]:
OUTPUT_PARTITIONED     = OUTPUT_PATH.parent / "dim_fornecedor_parts"
OUTPUT_PARTITIONED_STR = str(OUTPUT_PARTITIONED).replace("\\", "/")

# Clean previous partial run
if OUTPUT_PARTITIONED.exists():
    shutil.rmtree(OUTPUT_PARTITIONED)
OUTPUT_PARTITIONED.mkdir()

ufs = [r[0] for r in con.execute(
    "SELECT DISTINCT uf FROM t_dim_base ORDER BY uf"
).fetchall()]

print(f"UFs found: {len(ufs)} — {ufs}")
print()

t_total     = datetime.now()
grand_total = 0
errors      = []

for uf in ufs:
    t_uf     = datetime.now()
    out_file = f"{OUTPUT_PARTITIONED_STR}/uf={uf}.parquet"

    try:
        con.execute(f"""
            COPY (
                SELECT
                    -- Deterministic surrogate key: MD5(natural_key || valid_from)
                    -- No ORDER BY needed — no sort blocking operator
                    md5(cnpj_normalized || '|' || DATE '{VALID_FROM}')::VARCHAR AS supplier_sk,
                    cnpj_normalized,
                    razao_social,
                    porte,
                    situacao_cadastral,
                    natureza_juridica_desc,
                    uf,
                    municipio_desc,
                    tem_sancao,
                    _attr_hash,
                    DATE '{VALID_FROM}'        AS valid_from,
                    DATE '{VALID_TO}'          AS valid_to,
                    TRUE                       AS is_current,
                    '{LOADED_AT}'::TIMESTAMPTZ AS _loaded_at
                FROM t_dim_base
                WHERE uf = '{uf}'
            ) TO '{out_file}' (FORMAT PARQUET)
        """)

        n            = con.execute(f"SELECT COUNT(*) FROM read_parquet('{out_file}')").fetchone()[0]
        grand_total += n
        elapsed_uf   = (datetime.now() - t_uf).total_seconds()
        print(f"  uf={uf:<3}  rows={n:>10,}  time={elapsed_uf:>5.1f}s  cumulative={grand_total:,}")

    except Exception as e:
        errors.append((uf, str(e)[:120]))
        print(f"  uf={uf:<3}  ERROR: {str(e)[:120]}")

elapsed_total = (datetime.now() - t_total).total_seconds()
print()
print(f"Loop complete: {grand_total:,} rows in {elapsed_total:.0f}s ({elapsed_total/60:.1f} min)")

if errors:
    print()
    print("ERRORS:")
    for uf, msg in errors:
        print(f"  uf={uf}: {msg}")

## Step 5 — Consolidate parts into final Parquet file

In [ ]:
parts_glob = f"{OUTPUT_PARTITIONED_STR}/*.parquet"

print("Consolidating parts into single Parquet file...")
t0 = datetime.now()

con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{parts_glob}')
    ) TO '{OUTPUT_PATH_STR}' (FORMAT PARQUET)
""")

# Remove intermediate parts
shutil.rmtree(OUTPUT_PARTITIONED)
print(f"Removed parts dir: {OUTPUT_PARTITIONED}")

# Remove staging table from DuckDB
con.execute("DROP TABLE IF EXISTS t_dim_base")
print("Staging table t_dim_base dropped.")

file_size_mb = OUTPUT_PATH.stat().st_size / 1_048_576
elapsed      = (datetime.now() - t0).total_seconds()
print()
print(f"Written : {OUTPUT_PATH}")
print(f"Size    : {file_size_mb:.2f} MB")
print(f"Time    : {elapsed:.0f}s")

## Step 6 — Validation

All checks query the final Parquet directly via DuckDB — no DataFrame loaded into RAM.


In [ ]:
checks = []

total = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')"
).fetchone()[0]

# CHECK 1 — row count matches unique CNPJs in silver_identidade
expected = con.execute(
    "SELECT COUNT(DISTINCT cnpj_normalized) FROM v_identidade"
).fetchone()[0]
checks.append((
    "Row count = unique CNPJs (initial load)",
    total == expected,
    f"{total:,} (expected {expected:,})"
))

# CHECK 2 — supplier_sk unique
unique_sk = con.execute(
    f"SELECT COUNT(DISTINCT supplier_sk) FROM read_parquet('{OUTPUT_PATH_STR}')"
).fetchone()[0]
checks.append(("supplier_sk unique", unique_sk == total, f"{unique_sk:,} unique / {total:,} total"))

# CHECK 3 — no nulls in key columns
null_key = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE supplier_sk    IS NULL
       OR cnpj_normalized IS NULL
       OR _attr_hash      IS NULL
       OR valid_from      IS NULL
       OR valid_to        IS NULL
""").fetchone()[0]
checks.append(("No nulls in key columns", null_key == 0, f"{null_key} nulls found"))

# CHECK 4 — is_current = True for all rows (initial load)
not_current = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}') WHERE is_current = false"
).fetchone()[0]
checks.append((
    "All rows is_current=True (initial load)",
    not_current == 0,
    f"{not_current} rows with False"
))

# CHECK 5 — valid_to = 9999-12-31 for all rows
wrong_vt = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE CAST(valid_to AS VARCHAR) != '{VALID_TO}'
""").fetchone()[0]
checks.append(("valid_to = 9999-12-31 (initial load)", wrong_vt == 0, f"{wrong_vt} rows differ"))

# CHECK 6 — todo cnpj com tem_sancao=True na dim existe em CEIS∪CNEP (referential check)
# Não comparamos totais: 73 CNPJs existem em CEIS/CNEP mas não na Receita Federal
# — correto por design, a dim só contém fornecedores com registro cadastral

false_positives = con.execute(f"""
    WITH sancoes AS (
        SELECT cnpj_normalized FROM v_ceis
        UNION
        SELECT cnpj_normalized FROM v_cnep
    )
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}') d
    LEFT JOIN sancoes s USING (cnpj_normalized)
    WHERE d.tem_sancao = true
      AND s.cnpj_normalized IS NULL
""").fetchone()[0]

checks.append((
    "tem_sancao=True sem registro em CEIS∪CNEP",
    false_positives == 0,
    f"{false_positives} false positives"
))

# CHECK 7 — supplier_sk deterministic: MD5 of cnpj_normalized || valid_from
# Spot-check: recompute for 5 random rows and compare
mismatch = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE supplier_sk != md5(cnpj_normalized || '|' || CAST(valid_from AS VARCHAR))
    LIMIT 100
""").fetchone()[0]
checks.append(("supplier_sk = MD5(cnpj||valid_from)", mismatch == 0, f"{mismatch} mismatches"))

# ── Report ────────────────────────────────────────────────────────────────────
print(f"\n{'CHECK':<50} {'STATUS':<8} DETAIL")
print("-" * 85)
all_pass = True
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    print(f"{name:<50} [{status}]   {detail}")
    if not passed:
        all_pass = False

print()
if all_pass:
    print("All checks PASSED — dim_fornecedor ready.")
else:
    raise AssertionError("One or more validation checks FAILED — review output above.")

# Sample rows
print()
print("Sample rows (uf=SP, tem_sancao=False):")
sample = con.execute(f"""
    SELECT supplier_sk, cnpj_normalized, razao_social,
           situacao_cadastral, porte, tem_sancao, valid_from, valid_to
    FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE uf = 'SP' AND tem_sancao = false
    LIMIT 3
""").df()
print(sample.to_string(index=False))

print()
print("Sample rows (tem_sancao=True):")
sample_s = con.execute(f"""
    SELECT supplier_sk, cnpj_normalized, razao_social,
           situacao_cadastral, porte, tem_sancao, valid_from, valid_to
    FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE tem_sancao = true
    LIMIT 3
""").df()
print(sample_s.to_string(index=False))